# USAGE OF THE DIFFUSION MODEL

In [ ]:
%run init_notebook.py

import torch.nn as nn
from torchvision.datasets import FashionMNIST, QMNIST, KMNIST
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import time
import json
from tqdm import tqdm  # import the class directly

import torchaudio.transforms as T
from src.dataset import NSynth
import torch
from src.diffusion import *
from src.utils.models import adjust_shape, compute_magnitude_and_phase, compute_magnitude_and_phase_sin_cos
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

root = r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\diffusion"
paths = {
    'minst' : {
        'root': root + r"\minst",
        'model': root + r"\minst\model.pth",
        'object': root + r"\minst\object.pth",
        'config': root + r"\minst\config.json",
        'scheduler': root + r"\minst\scheduler.pth",
        'checkpoint': root + r"\minst\checkpoint.pth",
        'dataset' : r"C:\Users\Articuno\Desktop\TFG-info\data\mnist"
    },
    'audio' : {
        'root': root + r"\audio",
        'model': root + r"\audio\model.pth",
        'config': root + r"\audio\config.json",
        'object': root + r"\audio\object.pth",
        'scheduler': root + r"\audio\scheduler.pth",
        'checkpoint': root + r"\audio\checkpoint.pth"
    },
    'latent' : {
        'root': root + r"\latent",
        'model': root + r"\latent\model.pth",
        'object': root + r"\latent\object.pth",
        'config': root + r"\latent\config.json",
        'scheduler': root + r"\latent\scheduler.pth",
        'checkpoint': root + r"\latent\checkpoint.pth",
        'dataset_training' : r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\diffusion\latent\dataset\training.pt",
        'training_norm_params' :  r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\diffusion\latent\dataset\training_params.pt",
    }
}

def show_loss_plot(losses):
    epochs_list = [d['epoch'] for d in losses]
    loss_list   = [d['loss']  for d in losses]
    lr_list     = [d['lr']    for d in losses]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

    ax1.plot(epochs_list, loss_list)
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss')
    ax1.grid(True)

    ax2.plot(epochs_list, lr_list, color='orange')
    ax2.set_ylabel('Learning Rate')
    ax2.set_xlabel('Epoch')
    ax2.set_title('Learning Rate Schedule')
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

## LOAD MODELS

In [ ]:
from src.diffusion.setup_audio import setup_audio_model
from src.diffusion.setup_minst import setup_minst_model
from src.diffusion.setup_latent import setup_latent_model

def load_audio_model():
    config = {}
    config = torch.load(paths['audio']['config'])
    model, scheduler, _ = setup_audio_model(
        timesteps=config['timesteps'],
        channels=config['channels'],
        norm_groups=config['norm_groups'],
        emb_dim=config['emb_dim'],
        n_ftt=config['n_ftt'],
        hop_length=config['hop_length'],
        win_length=config['win_length']
    )
    model.load_state_dict(torch.load(paths['audio']['model']))
    scheduler.load_state_dict(torch.load(paths['audio']['scheduler']))
    return model, scheduler

def load_minst_model():
    config = {}
    config = torch.load(paths['minst']['config'])
    model, scheduler, _ = setup_minst_model(
        timesteps=config['timesteps'],
        channels=config['channels'],
        norm_groups=config['norm_groups'],
        emb_dim=config['emb_dim']
    )
    model.load_state_dict(torch.load(paths['minst']['model']))
    scheduler.load_state_dict(torch.load(paths['minst']['scheduler']))
    return model, scheduler

def load_latent_model():
    config = {}
    config = torch.load(paths['latent']['config'])
    model, scheduler, _ = setup_latent_model(
        timesteps=config['timesteps'],
        emb_dim=config['emb_dim'],
        hidden_dim=config['hidden_dim'],
        latent_dim=config['latent_dim'],
        deep=config['deep'],
        increase_dim=config['increase_dim']
    )
    model.load_state_dict(torch.load(paths['latent']['model']))
    scheduler.load_state_dict(torch.load(paths['latent']['scheduler']))
    return model, scheduler

## SAMPLE IMAGES

In [ ]:
def sample_raw_img(model, scheduler, image_size=(1, 28, 28), num_images=1):
    model.eval() # NS
    # with torch.no_grad():
    
    x = torch.randn(num_images, *image_size, device=device) # imagen de ruidio inicial
    T = scheduler.alpha_bar.shape[0]
    
    betas = scheduler.beta
    alphas = scheduler.alpha
    alpha_bars = scheduler.alpha_bar
    with torch.no_grad():

        for t in reversed(range(T)):
            t_batch = torch.full((num_images,), t, device=device, dtype=torch.long)

            # predicción de ruido
            e_pred = model(x, t_batch)
            
            beta_t = betas[t]
            alpha_t = alphas[t]
            alpha_bar_t = alpha_bars[t]
            
            # beta_t = beta_t.view(1,1,1,1)
            # alpha_t = alpha_t.view(1,1,1,1)
            # alpha_bar_t = alpha_bar_t.view(1,1,1,1)


            # Coeficientes DDPM
            coef1 = 1 / torch.sqrt(alpha_t)
            coef2 = beta_t / torch.sqrt(1 - alpha_bar_t)

            mu = coef1 * (x - coef2 * e_pred)

            if t > 0:
                noise = torch.randn_like(x)
                sigma_t = torch.sqrt(beta_t)
                x = mu + sigma_t * noise
            else:
                x = mu

        x = x.clamp(0, 1).cpu()
        
    return x

def denoise_img(model, scheduler, noisy_img, num_steps=300, step_imgs=10):
    model.eval()
    xs = []
    x = noisy_img.to(device)
    T = scheduler.alpha_bar.shape[0]
    
    betas = scheduler.beta
    alphas = scheduler.alpha
    alpha_bars = scheduler.alpha_bar
    
    with torch.no_grad():
        for t in reversed(range(T)):
            t_batch = torch.full((x.size(0),), t, device=device, dtype=torch.long)

            e_pred = model(x, t_batch)
            
            beta_t = betas[t]
            alpha_t = alphas[t]
            alpha_bar_t = alpha_bars[t]

            coef1 = 1 / torch.sqrt(alpha_t)
            coef2 = beta_t / torch.sqrt(1 - alpha_bar_t)

            mu = coef1 * (x - coef2 * e_pred)

            if t > 0:
                noise = torch.randn_like(x)
                sigma_t = torch.sqrt(beta_t)
                x = mu + sigma_t * noise
            else:
                x = mu
                
            if t % (T // step_imgs) == 0:
                xs.append(x.clamp(0, 1).cpu())

        x = x.clamp(0, 1).cpu()
        
    return x, xs
    
def show_imgs(xs, cols=5):
    rows = math.ceil(len(xs) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2))
    axes = axes.flatten()

    for i in range(len(xs)):
        axes[i].imshow(xs[i][0].detach().cpu().numpy(), cmap='gray')
        axes[i].axis('off')

    # Ocultar ejes sobrantes si n no es múltiplo de cols
    for j in range(len(xs), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()